# Watermark Robustness Experiment

## 실험 목표
1. **semantic_wm 데이터셋**으로 VAE 학습
2. **Zero-shot 원본 이미지** (demo_war/example/input/0.png) 사용
3. 원본 이미지 → CLIP → Latent → Watermark 추출
4. VINE으로 워터마크 삽입
5. InstructPix2Pix로 폭력적 변형
6. 변형된 이미지 CLIP vs 워터마크 복원 CLIP 비교

## 파이프라인
```
Training: semantic_wm dataset → VAE(512D → 100D)
Testing:  0.png → CLIP(512D) → Latent(100D) → Watermark(100bit)
                  ↓
          VINE Embedding → Transformed Image (InstructPix2Pix)
                  ↓
          Comparison: Transformed CLIP vs Watermark→Decoder→CLIP
```

## 1. 환경 설정 및 패키지 설치

In [ ]:
# Core ML libraries
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Computer Vision & Image Processing
# !pip install -q numpy==1.26.4 scipy==1.12.0 pillow==10.4.0 opencv-python==4.8.1.78
# !pip install -q scikit-image scikit-learn matplotlib seaborn

!pip install -q numpy>=2.0 scipy>=1.13 pillow==10.4.0 opencv-python
!pip install -q scikit-image scikit-learn matplotlib seaborn

In [ ]:
# Deep Learning utilities
!pip install -q einops==0.8.0 timm==0.9.12 lpips clean-fid kornia
!pip install -q tqdm easydict sentencepiece
# !pip install -q fsspec>=2025.3.0

In [ ]:
# Transformers and CLIP
!pip install -q transformers==4.45.2 open-clip-torch==2.26.1

In [ ]:
# Diffusion models (for InstructPix2Pix)
!pip install -q diffusers==0.30.0 accelerate==0.34.2 peft==0.17.1 datasets==2.20.0

In [ ]:
print("✅ All packages installed successfully!")

In [ ]:
# 런타임 재시작
import os
os.kill(os.getpid(), 9)

## 2. 라이브러리 Import

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from torchvision import transforms
from tqdm import tqdm
import warnings
import time
import gc

warnings.filterwarnings('ignore')

# GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## 4. semantic_wm 데이터셋 압축 해제

In [ ]:
# semantic_wm 데이터셋 압축 해제
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("압축 해제 중...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("✅ 데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
print("\n=== 데이터셋 구조 ===")
!ls -R /content/semantic_wm/dataset/train/

## 5. CLIP 모델 로드

In [ ]:
# CLIP 모델 로드
model_name = "openai/clip-vit-base-patch32"
print(f"Loading CLIP model: {model_name}")

clip_model = CLIPModel.from_pretrained(model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(model_name)

print(f"✅ CLIP 모델 로드 완료")
print(f"   - Model: {model_name}")
print(f"   - Embedding dimension: 512D")

## 6. 데이터 로드 함수 정의

In [ ]:
def load_images_from_category(base_path, category, max_images=None):
    """카테고리별 이미지 경로 로드"""
    category_path = os.path.join(base_path, category)
    image_files = []

    for file in os.listdir(category_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            image_files.append(os.path.join(category_path, file))

    if max_images:
        image_files = image_files[:max_images]

    return image_files

def extract_clip_embeddings(image_paths, model, processor, device, batch_size=32):
    """CLIP 이미지 임베딩 추출"""
    embeddings = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting CLIP embeddings"):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []

            for img_path in batch_paths:
                try:
                    image = Image.open(img_path).convert('RGB')
                    batch_images.append(image)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
                    continue

            if batch_images:
                inputs = processor(images=batch_images, return_tensors="pt", padding=True)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                image_features = model.get_image_features(**inputs)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                embeddings.append(image_features.cpu().numpy())

    return np.vstack(embeddings)

print("✅ 데이터 로드 함수 정의 완료")

## 7. semantic_wm 데이터셋 로드 및 CLIP 임베딩 추출

In [ ]:
# semantic_wm 데이터셋 로드
train_path = '/content/semantic_wm/dataset/train'
categories = ['normal', 'violence', 'sexual']

all_embeddings = []
all_labels = []
category_stats = {}

# 카테고리당 최대 이미지 수 설정 (메모리 절약)
max_images_per_category = None

print("\n" + "="*80)
print("Training Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(train_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    all_embeddings.append(embeddings)
    all_labels.extend([category] * len(embeddings))
    category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
all_embeddings = np.vstack(all_embeddings)
all_labels = np.array(all_labels)

print("\n" + "="*80)
print("Training Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {all_embeddings.shape}")
print(f"전체 레이블 수: {len(all_labels)}")
print("\n카테고리별 통계:")
for cat, count in category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(all_labels)*100:.1f}%)")
print("="*80)

In [ ]:
# CLIP Embedding Space t-SNE 계산
print("\nCLIP Embedding Space t-SNE 계산 중...")
clip_tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
clip_embeddings_2d = clip_tsne.fit_transform(all_embeddings)

print(f"✅ CLIP t-SNE 완료: {clip_embeddings_2d.shape}")
print(f"   - 원본 차원: {all_embeddings.shape[1]}D (CLIP)")
print(f"   - 축소 차원: 2D (t-SNE)")

In [ ]:
# CLIP Embedding Space 시각화
plt.figure(figsize=(14, 10))

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for category in categories:
    mask = all_labels == category
    plt.scatter(
        clip_embeddings_2d[mask, 0],
        clip_embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=f'{category.capitalize()}',
        alpha=0.6,
        s=50,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('CLIP Embedding Space t-SNE Visualization (512D → 2D)',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=12, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('/content/clip_embedding_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ CLIP Embedding t-SNE 시각화 저장: /content/clip_embedding_tsne.png")

In [ ]:
# CLIP Embedding 통계 분석
print("\n" + "="*80)
print("CLIP Embedding Space 분석")
print("="*80)

# 카테고리별 중심 계산
category_centers = {}
for category in categories:
    mask = all_labels == category
    category_embeddings = all_embeddings[mask]
    category_center = category_embeddings.mean(axis=0)
    category_centers[category] = category_center

    print(f"\n[{category.capitalize()}]")
    print(f"  - 샘플 수: {np.sum(mask)}")
    print(f"  - 평균 norm: {np.linalg.norm(category_center):.4f}")

# 카테고리 간 거리 계산
print("\n카테고리 간 코사인 유사도:")
from itertools import combinations
for cat1, cat2 in combinations(categories, 2):
    center1 = category_centers[cat1]
    center2 = category_centers[cat2]
    similarity = np.dot(center1, center2) / (np.linalg.norm(center1) * np.linalg.norm(center2))
    print(f"  - {cat1.capitalize()} vs {cat2.capitalize()}: {similarity:.4f}")

print("="*80)

## 8. VAE 모델 정의

In [ ]:
# VAE 모델 정의
class CLIPCompressionVAE(nn.Module):
    def __init__(self, input_dim=512, latent_dim=100):
        super().__init__()

        # Encoder: CLIP(512) → Latent(latent_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: Latent(latent_dim) → CLIP'(512)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        recon = self.decoder(z)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

print("✅ VAE 모델 정의 완료")
print(f"   - Input: 512D (CLIP)")
print(f"   - Latent: 100D")
print(f"   - Output: 512D (Reconstructed CLIP)")

## 9. 데이터셋 준비

In [ ]:
# 데이터셋 클래스 정의
class CLIPEmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

# Train/Val split
train_embeddings, val_embeddings, train_labels, val_labels = train_test_split(
    all_embeddings, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

train_dataset = CLIPEmbeddingDataset(train_embeddings, train_labels)
val_dataset = CLIPEmbeddingDataset(val_embeddings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print("✅ 데이터셋 준비 완료")
print(f"   - 학습 데이터: {len(train_dataset)} samples")
print(f"   - 검증 데이터: {len(val_dataset)} samples")
print(f"   - Batch size: 64")

## 10. VAE 손실 함수 및 학습 함수 정의

In [ ]:
# VAE 손실 함수
def vae_loss(recon_x, x, mu, logvar, beta=0.01):
    """
    VAE Loss = Reconstruction Loss + KLD Loss

    Args:
        recon_x: 복원된 CLIP embedding
        x: 원본 CLIP embedding
        mu: latent mean
        logvar: latent log variance
        beta: KLD weight

    Returns:
        total_loss, recon_loss, kld_loss
    """
    # Reconstruction loss (Cosine similarity)
    cosine_sim = F.cosine_similarity(recon_x, x, dim=1).mean()
    recon_loss = 1 - cosine_sim

    # KLD loss
    kld_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    total_loss = recon_loss + beta * kld_loss

    return total_loss, recon_loss, kld_loss

print("✅ VAE 손실 함수 정의 완료")

In [ ]:
# VAE 학습 함수 (tqdm 포함)
def train_vae(model, train_loader, val_loader, epochs=50, latent_dim=100, beta=0.01):
    """
    VAE 학습 함수

    Args:
        model: VAE 모델
        train_loader: 학습 데이터 로더
        val_loader: 검증 데이터 로더
        epochs: 학습 에포크 수
        latent_dim: latent dimension
        beta: KLD weight

    Returns:
        model: 학습된 모델
        history: 학습 히스토리 (loss, metrics)
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    best_val_loss = float('inf')
    best_model_state = None

    # Loss tracking for visualization
    train_losses = []
    val_losses = []
    train_recon_losses = []
    train_kld_losses = []
    val_recon_losses = []
    val_kld_losses = []
    val_cosine_similarities = []

    print(f"\\n{'='*80}")
    print(f"VAE 학습 시작 (Latent {latent_dim}D)")
    print(f"{'='*80}")
    print(f"학습 설정:")
    print(f"  - Epochs: {epochs}")
    print(f"  - Learning rate: 0.001")
    print(f"  - Beta (KLD weight): {beta}")
    print(f"  - Optimizer: Adam")
    print(f"  - Scheduler: ReduceLROnPlateau")
    print(f"{'='*80}\\n")

    for epoch in range(epochs):
        # ========== 학습 ==========
        model.train()
        train_loss = 0
        train_recon = 0
        train_kld = 0

        # tqdm으로 학습 진행률 표시
        train_pbar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}] Train", ncols=100)
        for embeddings, _ in train_pbar:
            embeddings = embeddings.to(device)

            optimizer.zero_grad()
            recon, mu, logvar, z = model(embeddings)

            loss, recon_l, kld_l = vae_loss(recon, embeddings, mu, logvar, beta)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_l.item()
            train_kld += kld_l.item()

            # tqdm 진행바에 현재 loss 표시
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'recon': f'{recon_l.item():.4f}',
                'kld': f'{kld_l.item():.4f}'
            })

        train_loss /= len(train_loader)
        train_recon /= len(train_loader)
        train_kld /= len(train_loader)

        # ========== 검증 ==========
        model.eval()
        val_loss = 0
        val_recon = 0
        val_kld = 0
        val_cosine_sims = []

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{epochs}] Val  ", ncols=100)
            for embeddings, _ in val_pbar:
                embeddings = embeddings.to(device)
                recon, mu, logvar, z = model(embeddings)

                loss, recon_l, kld_l = vae_loss(recon, embeddings, mu, logvar, beta)
                val_loss += loss.item()
                val_recon += recon_l.item()
                val_kld += kld_l.item()

                cos_sim = F.cosine_similarity(recon, embeddings, dim=1)
                val_cosine_sims.extend(cos_sim.cpu().numpy())

                val_pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        val_loss /= len(val_loader)
        val_recon /= len(val_loader)
        val_kld /= len(val_loader)
        avg_cosine_sim = np.mean(val_cosine_sims)

        # Loss 기록
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_recon_losses.append(train_recon)
        train_kld_losses.append(train_kld)
        val_recon_losses.append(val_recon)
        val_kld_losses.append(val_kld)
        val_cosine_similarities.append(avg_cosine_sim)

        scheduler.step(val_loss)

        # Best model 저장
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            print(f"  ✓ New best model! Val Loss: {best_val_loss:.4f}")

        # Epoch 요약
        print(f"\\nEpoch [{epoch+1}/{epochs}] Summary:")
        print(f"  Train Loss: {train_loss:.4f} (Recon: {train_recon:.4f}, KLD: {train_kld:.4f})")
        print(f"  Val Loss: {val_loss:.4f} | Val Cosine Sim: {avg_cosine_sim:.4f}")
        print(f"  Best Val Loss: {best_val_loss:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
        print()

    # Best model 로드
    model.load_state_dict(best_model_state)
    print(f"\\n{'='*80}")
    print(f"✅ VAE 학습 완료!")
    print(f"{'='*80}")
    print(f"  - Best Val Loss: {best_val_loss:.4f}")
    print(f"  - Final Train Loss: {train_losses[-1]:.4f}")
    print(f"  - Final Cosine Sim: {val_cosine_similarities[-1]:.4f}")
    print(f"{'='*80}\\n")

    # Loss 시각화를 위한 히스토리 반환
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_recon_losses': train_recon_losses,
        'train_kld_losses': train_kld_losses,
        'val_recon_losses': val_recon_losses,
        'val_kld_losses': val_kld_losses,
        'val_cosine_similarities': val_cosine_similarities
    }

    return model, history

print("✅ VAE 학습 함수 정의 완료")

## 11. VAE 학습 실행

In [ ]:
# VAE 모델 생성 및 학습
vae_model = CLIPCompressionVAE(input_dim=512, latent_dim=100)
vae_model, training_history = train_vae(
    vae_model,
    train_loader,
    val_loader,
    epochs=50,
    latent_dim=100,
    beta=0.01
)

## 12. 학습 Loss 시각화

In [ ]:
# Loss 시각화
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Total Loss (Train & Val)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(training_history['train_losses'], label='Train Loss', linewidth=2, color='#3498db', marker='o', markersize=3)
ax1.plot(training_history['val_losses'], label='Val Loss', linewidth=2, color='#e74c3c', marker='s', markersize=3)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Total Loss', fontsize=12)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 2. Reconstruction & KLD Loss (Dual Y-axis)
ax2 = fig.add_subplot(gs[0, 1])
ax2_twin = ax2.twinx()

# Reconstruction Loss (왼쪽 y축)
line1 = ax2.plot(training_history['train_recon_losses'], label='Train Recon', linewidth=2, color='#e74c3c', marker='o', markersize=3)
line2 = ax2.plot(training_history['val_recon_losses'], label='Val Recon', linewidth=2, color='#e74c3c', linestyle='--', marker='s', markersize=3, alpha=0.7)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Reconstruction Loss', fontsize=12, color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

# KLD Loss (오른쪽 y축)
line3 = ax2_twin.plot(training_history['train_kld_losses'], label='Train KLD', linewidth=2, color='#3498db', marker='^', markersize=3)
line4 = ax2_twin.plot(training_history['val_kld_losses'], label='Val KLD', linewidth=2, color='#3498db', linestyle='--', marker='v', markersize=3, alpha=0.7)
ax2_twin.set_ylabel('KLD Loss', fontsize=12, color='#3498db')
ax2_twin.tick_params(axis='y', labelcolor='#3498db')

# 범례 통합
lines = line1 + line2 + line3 + line4
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, fontsize=10, loc='upper right')
ax2.set_title('Reconstruction vs KLD Loss (Dual Y-axis)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Validation Cosine Similarity
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(training_history['val_cosine_similarities'], label='Val Cosine Sim', linewidth=2, color='#2ecc71', marker='o', markersize=4)
ax3.fill_between(range(len(training_history['val_cosine_similarities'])),
                  training_history['val_cosine_similarities'],
                  alpha=0.3, color='#2ecc71')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Cosine Similarity', fontsize=12)
ax3.set_title('Validation Cosine Similarity', fontsize=14, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)
if max(training_history['val_cosine_similarities']) - min(training_history['val_cosine_similarities']) < 0.1:
    ax3.set_ylim([0.9, 1.0])

# 4. Loss Components Breakdown (Stacked Area)
ax4 = fig.add_subplot(gs[1, 0])
epochs = range(len(training_history['train_recon_losses']))
ax4.fill_between(epochs, 0, training_history['train_recon_losses'], label='Recon Loss', alpha=0.7, color='#e74c3c')
ax4.fill_between(epochs, training_history['train_recon_losses'],
                  [r + k for r, k in zip(training_history['train_recon_losses'], training_history['train_kld_losses'])],
                  label='KLD Loss', alpha=0.7, color='#3498db')
ax4.set_xlabel('Epoch', fontsize=12)
ax4.set_ylabel('Loss', fontsize=12)
ax4.set_title('Training Loss Components (Stacked)', fontsize=14, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3)

# 5. Loss Comparison Table
ax5 = fig.add_subplot(gs[1, 1])
loss_data = [
    ['Initial Train Loss', f"{training_history['train_losses'][0]:.4f}"],
    ['Final Train Loss', f"{training_history['train_losses'][-1]:.4f}"],
    ['Best Val Loss', f"{min(training_history['val_losses']):.4f}"],
    ['Final Recon Loss', f"{training_history['train_recon_losses'][-1]:.4f}"],
    ['Final KLD Loss', f"{training_history['train_kld_losses'][-1]:.4f}"],
    ['Final Cosine Sim', f"{training_history['val_cosine_similarities'][-1]:.4f}"],
]
ax5.axis('tight')
ax5.axis('off')
table = ax5.table(cellText=loss_data, colLabels=['Metric', 'Value'],
                  cellLoc='left', loc='center',
                  colWidths=[0.65, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.0)
for i in range(len(loss_data) + 1):
    if i == 0:
        table[(i, 0)].set_facecolor('#3498db')
        table[(i, 1)].set_facecolor('#3498db')
        table[(i, 0)].set_text_props(weight='bold', color='white')
        table[(i, 1)].set_text_props(weight='bold', color='white')
    else:
        table[(i, 0)].set_facecolor('#ecf0f1')
        table[(i, 1)].set_facecolor('#ecf0f1')
ax5.set_title('Training Summary', fontsize=14, fontweight='bold', pad=20)

# 6. Loss Reduction Rate
ax6 = fig.add_subplot(gs[1, 2])
initial_train = training_history['train_losses'][0]
reduction_rate = [(initial_train - loss) / initial_train * 100 for loss in training_history['train_losses']]
ax6.plot(reduction_rate, linewidth=2, color='#9b59b6', marker='o', markersize=3)
ax6.fill_between(range(len(reduction_rate)), 0, reduction_rate, alpha=0.3, color='#9b59b6')
ax6.set_xlabel('Epoch', fontsize=12)
ax6.set_ylabel('Loss Reduction (%)', fontsize=12)
ax6.set_title('Training Loss Reduction Rate', fontsize=14, fontweight='bold')
ax6.grid(True, alpha=0.3)
ax6.axhline(y=0, color='gray', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('/content/vae_training_loss_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Loss 시각화 저장: /content/vae_training_loss_visualization.png")

## 13. Latent Space 추출 및 t-SNE 시각화

In [ ]:
# Latent Space 추출
print("Latent Space 추출 중...")
vae_model.eval()
with torch.no_grad():
    all_embeddings_tensor = torch.FloatTensor(all_embeddings).to(device)
    all_latent_mu, _ = vae_model.encode(all_embeddings_tensor)
    all_latent_vectors = all_latent_mu.cpu().numpy()

print(f"✅ Latent vectors shape: {all_latent_vectors.shape}")
print(f"   - Mean: {all_latent_vectors.mean():.4f}")
print(f"   - Std: {all_latent_vectors.std():.4f}")
print(f"   - Min: {all_latent_vectors.min():.4f}")
print(f"   - Max: {all_latent_vectors.max():.4f}")

In [ ]:
# Latent Space t-SNE
print("\\nLatent Space t-SNE 계산 중...")
latent_tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
latent_embeddings_2d = latent_tsne.fit_transform(all_latent_vectors)

print(f"✅ Latent t-SNE 완료: {latent_embeddings_2d.shape}")

In [ ]:
# Latent Space 시각화
plt.figure(figsize=(14, 10))

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for category in categories:
    mask = all_labels == category
    plt.scatter(
        latent_embeddings_2d[mask, 0],
        latent_embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=f'{category.capitalize()}',
        alpha=0.6,
        s=50,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('Latent Space t-SNE Visualization (100D → 2D)',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=12, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('/content/latent_space_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Latent Space t-SNE 시각화 저장: /content/latent_space_tsne.png")

## 14. VINE 설치 및 모델 로드

In [ ]:
# VINE repository 클론
!git clone https://github.com/Shilin-LU/VINE.git
import sys
sys.path.append("/content/VINE")

print("✅ VINE 설치 완료")

In [ ]:
# VINE 모델 로드
from vine.src.vine_turbo import VINE_Turbo
from vine.src.stega_encoder_decoder import CustomConvNeXt
from accelerate.utils import set_seed

set_seed(42)

print("VINE 모델 로드 중...")
watermark_encoder = VINE_Turbo.from_pretrained("Shilin-LU/VINE-B-Enc").to(device)
vine_decoder = CustomConvNeXt.from_pretrained("Shilin-LU/VINE-B-Dec").to(device)

print("✅ VINE 모델 로드 완료")
print("   - Encoder: VINE-B-Enc (100bit watermark embedding)")
print("   - Decoder: VINE-B-Dec (watermark extraction)")

## 15. Test Image 로드 및 준비 (demo_war의 0.png)

In [ ]:
import requests

# Base URL for test images
base_url = "https://raw.githubusercontent.com/Shilin-LU/VINE/main/example/input/"

# Local save directory
save_dir = "./example/input"
os.makedirs(save_dir, exist_ok=True)

# Download test images (0.png to 4.png)
file_list = [f"{i}.png" for i in range(5)]

print("Test 이미지 다운로드 중...")
for filename in file_list:
    file_path = os.path.join(save_dir, filename)

    if os.path.exists(file_path):
        print(f"  ✓ {filename} - 이미 존재함")
        continue

    url = base_url + filename
    response = requests.get(url)

    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"  ✓ {filename} - 다운로드 완료")
    else:
        print(f"  ✗ {filename} - 다운로드 실패 (status: {response.status_code})")

print("\n✅ Test 이미지 다운로드 완료")

In [ ]:
# 이미지를 정사각형으로 크롭하는 함수
def crop_to_square(image):
    """이미지를 중앙 기준으로 정사각형으로 크롭"""
    width, height = image.size
    min_side = min(width, height)

    left = (width - min_side) // 2
    top = (height - min_side) // 2
    right = left + min_side
    bottom = top + min_side

    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

print("✅ crop_to_square 함수 정의 완료")

In [ ]:
# Test image 로드 (0.png)
input_path = './example/input/0.png'

input_image_pil = Image.open(input_path).convert('RGB')

# 정사각형으로 크롭 (필요시)
if input_image_pil.size[0] != input_image_pil.size[1]:
    input_image_pil = crop_to_square(input_image_pil)
    print(f"✓ 이미지를 정사각형으로 크롭: {input_image_pil.size}")

size = input_image_pil.size

# Transform 정의
t_val_256 = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
])
t_val_512 = transforms.Compose([
    transforms.Resize(size, interpolation=transforms.InterpolationMode.BICUBIC),
])

# 256x256 버전 (VINE 입력용)
resized_img = t_val_256(input_image_pil)
resized_img = 2.0 * resized_img - 1.0  # [-1, 1] 정규화
resized_img = resized_img.unsqueeze(0).to(device)

# 512x512 버전 (원본 크기 유지)
input_image = transforms.ToTensor()(input_image_pil).unsqueeze(0).to(device)
input_image = 2.0 * input_image - 1.0  # [-1, 1] 정규화

print(f"\n✅ Test image 로드 및 전처리 완료")
print(f"   - Path: {input_path}")
print(f"   - Original size: {size}")
print(f"   - Resized (VINE): {resized_img.shape}")
print(f"   - Original (full): {input_image.shape}")

# Display test image
plt.figure(figsize=(8, 8))
plt.imshow(input_image_pil)
plt.axis('off')
plt.title('Original Test Image (0.png)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 16. Test Image의 CLIP Embedding 추출

In [ ]:
# Test image의 CLIP embedding 추출
clip_model.eval()
with torch.no_grad():
    inputs = clip_processor(images=input_image_pil, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    test_clip_embedding = clip_model.get_image_features(**inputs)
    test_clip_embedding = test_clip_embedding / test_clip_embedding.norm(dim=-1, keepdim=True)
    test_clip_embedding = test_clip_embedding.cpu().numpy()[0]

print(f"✅ Test image CLIP embedding 추출 완료")
print(f"   - Shape: {test_clip_embedding.shape}")
print(f"   - Norm: {np.linalg.norm(test_clip_embedding):.4f}")
print(f"   - Mean: {test_clip_embedding.mean():.4f}")
print(f"   - Std: {test_clip_embedding.std():.4f}")

## 17. VAE로 Latent 추출 및 Watermark 생성

In [ ]:
# Watermark 변환 클래스 정의
class LatentWatermarker:
    def __init__(self, latent_dim=100, watermark_bits=100):
        self.latent_dim = latent_dim
        self.watermark_bits = watermark_bits
        self.pca = None

    def fit(self, latent_vectors):
        """PCA 학습 (필요시)"""
        if self.latent_dim > self.watermark_bits:
            self.pca = PCA(n_components=self.watermark_bits)
            self.pca.fit(latent_vectors)
            print(f"✅ PCA 학습 완료: {self.latent_dim}D → {self.watermark_bits}D")
            print(f"   - 설명된 분산: {self.pca.explained_variance_ratio_.sum():.4f}")
        else:
            print(f"✅ Latent {self.latent_dim}D는 PCA 없이 직접 변환")

    def latent_to_watermark(self, latent_vector):
        """Latent → Watermark (100bit) 변환"""
        if self.pca is not None:
            if latent_vector.ndim == 1:
                latent_compressed = self.pca.transform(latent_vector.reshape(1, -1))[0]
            else:
                latent_compressed = self.pca.transform(latent_vector)
        else:
            latent_compressed = latent_vector

        # Sign-based quantization: positive → 1, negative → 0
        if latent_compressed.ndim == 1:
            watermark = (latent_compressed > 0).astype(int)
        else:
            watermark = (latent_compressed > 0).astype(int)

        return watermark

    def watermark_to_latent(self, watermark, latent_stats=None):
        """Watermark (100bit) → Latent 복원"""
        # 0/1 → -1/+1 변환
        latent_approx = watermark.astype(float) * 2 - 1

        # 통계 정보로 스케일 복원
        if latent_stats is not None:
            mean, std = latent_stats
            latent_approx = latent_approx * std + mean

        # PCA 역변환
        if self.pca is not None:
            if latent_approx.ndim == 1:
                latent_restored = self.pca.inverse_transform(latent_approx.reshape(1, -1))[0]
            else:
                latent_restored = self.pca.inverse_transform(latent_approx)
        else:
            latent_restored = latent_approx

        return latent_restored

print("✅ LatentWatermarker 클래스 정의 완료")

In [ ]:
# VAE Encoder로 Latent 추출
vae_model.eval()
with torch.no_grad():
    test_clip_tensor = torch.FloatTensor(test_clip_embedding).unsqueeze(0).to(device)
    test_latent_mu, test_latent_logvar = vae_model.encode(test_clip_tensor)
    test_latent = test_latent_mu.cpu().numpy()[0]

print(f"✅ Test image Latent (100D) 추출 완료")
print(f"   - Shape: {test_latent.shape}")
print(f"   - Mean: {test_latent.mean():.4f}")
print(f"   - Std: {test_latent.std():.4f}")
print(f"   - Min: {test_latent.min():.4f}")
print(f"   - Max: {test_latent.max():.4f}")

In [ ]:
# Watermarker 학습 및 Watermark 생성
print("\\nWatermarker 학습 중...")
watermarker = LatentWatermarker(latent_dim=100, watermark_bits=100)
watermarker.fit(all_latent_vectors)

# Test image의 Latent → Watermark 변환
test_watermark = watermarker.latent_to_watermark(test_latent)

print(f"\\n✅ Test image Watermark (100bit) 생성 완료")
print(f"   - Shape: {test_watermark.shape}")
watermark_str = ''.join(map(str, test_watermark))
print(f"   - Watermark: {watermark_str[:50]}")
print(f"                {watermark_str[50:]}")
print(f"   - Bit 분포: 1={np.sum(test_watermark)}/100, 0={100-np.sum(test_watermark)}/100")

## 18. VINE으로 Watermark를 이미지에 삽입 (demo_war flow)

## 13.5. Test 데이터셋 로드 및 워터마크 복원 분석

In [ ]:
# VAE로 생성된 워터마크를 VINE 형식으로 변환
vine_watermark = torch.tensor(test_watermark, dtype=torch.float).unsqueeze(0).to(device)
groundtruth_watermark = vine_watermark.clone()

print(f"\n✅ VINE Watermark 준비 완료")
print(f"   - Shape: {vine_watermark.shape}")
print(f"   - Device: {vine_watermark.device}")
print(f"   - Type: {vine_watermark.dtype}")
print(f"   - Bit 분포: 1={torch.sum(vine_watermark).item()}/100, 0={100-torch.sum(vine_watermark).item()}/100")

In [ ]:
# VINE Watermark Encoding
print("\n" + "="*80)
print("VINE Watermark Encoding 시작...")
print("="*80)

start_time = time.time()

watermark_encoder.eval()
with torch.no_grad():
    # VINE Watermark Encoding (demo_war.ipynb 방식)
    # secret 키워드 인자 사용
    encoded_image_256 = watermark_encoder(resized_img, secret=vine_watermark)

    # Residual 계산 및 원본 크기로 복원
    residual_256 = encoded_image_256 - resized_img  # 256x256
    residual_512 = t_val_512(transforms.ToPILImage()(residual_256.squeeze(0).cpu() * 0.5 + 0.5))
    residual_512 = transforms.ToTensor()(residual_512).unsqueeze(0).to(device)
    residual_512 = 2.0 * residual_512 - 1.0

    # 원본 이미지에 residual 추가
    encoded_img = residual_512 + input_image  # 512x512
    encoded_img = encoded_img * 0.5 + 0.5  # [-1,1] → [0,1]
    encoded_img = torch.clamp(encoded_img, min=0.0, max=1.0)

elapsed_time = time.time() - start_time

print(f"✅ VINE Watermark Encoding 완료")
print(f"   - 소요 시간: {elapsed_time:.2f}초")
print(f"   - Encoded image shape: {encoded_img.shape}")
print(f"   - Value range: [{encoded_img.min().item():.3f}, {encoded_img.max().item():.3f}]")
print("="*80 + "\n")

# GPU 메모리 정리
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Watermarked 이미지를 PIL로 변환 (이미 [0,1] 범위)
watermarked_pil = transforms.ToPILImage()(encoded_img.squeeze(0).cpu())

# Display: Original vs Watermarked
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(input_image_pil)
axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(watermarked_pil)
axes[1].set_title('Watermarked Image (VINE)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/original_vs_watermarked.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 이미지 비교 저장: /content/original_vs_watermarked.png")

## 19. InstructPix2Pix로 이미지 변형 (Violence)

In [ ]:
# InstructPix2Pix 파이프라인 로드
from diffusers import StableDiffusionInstructPix2PixPipeline, DDIMScheduler

print("InstructPix2Pix 모델 로드 중...")
instruct_pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
instruct_pipe.scheduler = DDIMScheduler.from_config(instruct_pipe.scheduler.config)
instruct_pipe.to(device)

print("✅ InstructPix2Pix 모델 로드 완료")

In [ ]:
# 이미지 편집 함수 (demo_war와 동일)
def edit_by_InstructPix2Pix(pipe, image, prompt, guidance=7.5, num_inference_steps=50):
    """
    InstructPix2Pix를 사용한 이미지 편집

    Args:
        pipe: InstructPix2Pix pipeline
        image: PIL Image
        prompt: 편집 프롬프트
        guidance: guidance scale
        num_inference_steps: 추론 스텝 수

    Returns:
        edited_image: PIL Image
    """
    edited_image = pipe(
        prompt=prompt,
        image=image,
        guidance_scale=guidance,
        image_guidance_scale=1.5,
        num_inference_steps=num_inference_steps,
    ).images[0]

    return edited_image

print("✅ 이미지 편집 함수 정의 완료")

In [ ]:
# 워터마크가 삽입된 이미지를 폭력적 장면으로 변형 (demo_war와 동일 프롬프트)
edit_prompt = "Transform the background into a war zone with smoke and destruction, make the child running away from the battlefield"

print("\\n" + "="*80)
print("이미지 편집 중...")
print("="*80)
print(f"Prompt: {edit_prompt}")
print()

edited_image = edit_by_InstructPix2Pix(
    instruct_pipe,
    watermarked_pil,
    edit_prompt,
    guidance=7.5,
    num_inference_steps=50
)

# Save edited image
edited_dir = '/content/edited_output'
os.makedirs(edited_dir, exist_ok=True)
edited_path = os.path.join(edited_dir, '0_wm_edit.png')
edited_image.save(edited_path)

print(f"\\n✅ 이미지 편집 완료")
print(f"   - Edited image saved: {edited_path}")
print("="*80)

In [ ]:
# Display: Original → Watermarked → Edited
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

axes[0].imshow(input_image_pil)
axes[0].set_title('1. Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(watermarked_pil)
axes[1].set_title('2. Watermarked Image (VINE)', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(edited_image)
axes[2].set_title('3. Edited Image (Violence)', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/image_transformation_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 이미지 변환 파이프라인 저장: /content/image_transformation_pipeline.png")

## 20. 편집된 이미지에서 Watermark 복원 (VINE Decoder)

In [ ]:
# 편집된 이미지에서 워터마크 추출
print("\\n" + "="*80)
print("편집된 이미지에서 워터마크 복원 중...")
print("="*80)

edited_image_tensor = t_val_256(edited_image).unsqueeze(0).to(device)

vine_decoder.eval()
with torch.no_grad():
    decoded_watermark = vine_decoder(edited_image_tensor)
    decoded_watermark = np.array(decoded_watermark[0].cpu().detach())
    decoded_watermark = np.round(decoded_watermark).astype(int)

# 원본 워터마크와 비교
original_watermark_list = test_watermark.tolist()
decoded_watermark_list = decoded_watermark.tolist()

same_bits = sum(x == y for x, y in zip(original_watermark_list, decoded_watermark_list))
bit_accuracy = same_bits / 100

print(f"\\n✅ 워터마크 복원 완료")
print("="*80)

In [ ]:
# 워터마크 복원 결과 출력 (demo_war 스타일)
print("\\n[Original Watermark (from VAE)]")
original_str = ''.join(map(str, original_watermark_list))
print(f"Bits: {original_str[:50]}")
print(f"      {original_str[50:]}")

print("\\n[Decoded Watermark (from Edited Image)]")
decoded_str = ''.join(map(str, decoded_watermark_list))
print(f"Bits: {decoded_str[:50]}")
print(f"      {decoded_str[50:]}")

print(f"\\n[Accuracy]")
print(f"Bit Accuracy: {bit_accuracy:.2%} ({same_bits}/100 bits)")

# 오류 위치 표시
errors = [i for i, (o, d) in enumerate(zip(original_watermark_list, decoded_watermark_list)) if o != d]
if len(errors) > 0:
    print(f"\\nError positions (first 20): {errors[:20]}" + (' ...' if len(errors) > 20 else ''))
    print(f"Total errors: {len(errors)} bits")
else:
    print(f"\\n✓ Perfect watermark recovery!")
print("="*80)

## 21. CLIP Embedding 비교 분석

In [ ]:
# 편집된 이미지의 CLIP embedding 추출
print("\\n편집된 이미지의 CLIP embedding 추출 중...")

clip_model.eval()
with torch.no_grad():
    inputs = clip_processor(images=edited_image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    edited_clip_embedding = clip_model.get_image_features(**inputs)
    edited_clip_embedding = edited_clip_embedding / edited_clip_embedding.norm(dim=-1, keepdim=True)
    edited_clip_embedding = edited_clip_embedding.cpu().numpy()[0]

print(f"✅ 편집된 이미지 CLIP embedding shape: {edited_clip_embedding.shape}")

In [ ]:
# Watermark → Latent → CLIP 복원
latent_stats = (all_latent_vectors.mean(axis=0), all_latent_vectors.std(axis=0))
test_latent_restored = watermarker.watermark_to_latent(test_watermark, latent_stats)

# VAE Decoder로 CLIP 복원
vae_model.eval()
with torch.no_grad():
    test_latent_restored_tensor = torch.FloatTensor(test_latent_restored).unsqueeze(0).to(device)
    test_clip_reconstructed = vae_model.decode(test_latent_restored_tensor)
    test_clip_reconstructed = test_clip_reconstructed.cpu().numpy()[0]

print(f"✅ Watermark에서 CLIP 복원 완료")
print(f"   - Reconstructed CLIP shape: {test_clip_reconstructed.shape}")

In [ ]:
# 편집된 이미지에서 추출한 Watermark → CLIP 복원
print("\n편집된 이미지의 Watermark에서 CLIP 복원 중...")

decoded_latent_restored = watermarker.watermark_to_latent(decoded_watermark, latent_stats)

vae_model.eval()
with torch.no_grad():
    decoded_latent_restored_tensor = torch.FloatTensor(decoded_latent_restored).unsqueeze(0).to(device)
    decoded_clip_reconstructed = vae_model.decode(decoded_latent_restored_tensor)
    decoded_clip_reconstructed = decoded_clip_reconstructed.cpu().numpy()[0]

print(f"✅ 편집된 이미지의 Watermark에서 CLIP 복원 완료")
print(f"   - Decoded & Reconstructed CLIP shape: {decoded_clip_reconstructed.shape}")

In [ ]:
# CLIP 유사도 비교 (4가지)
# 유사도 계산 (6가지)
clip_sim_original_vs_edited = np.dot(test_clip_embedding, edited_clip_embedding)
clip_sim_original_vs_reconstructed = np.dot(test_clip_embedding, test_clip_reconstructed)
clip_sim_edited_vs_reconstructed = np.dot(edited_clip_embedding, test_clip_reconstructed)

# 편집된 이미지의 워터마크 복원 CLIP 비교
clip_sim_original_vs_decoded_reconstructed = np.dot(test_clip_embedding, decoded_clip_reconstructed)
clip_sim_edited_vs_decoded_reconstructed = np.dot(edited_clip_embedding, decoded_clip_reconstructed)
clip_sim_original_reconstructed_vs_decoded_reconstructed = np.dot(test_clip_reconstructed, decoded_clip_reconstructed)

print("\\n" + "="*80)
print("CLIP EMBEDDING SIMILARITY ANALYSIS")
print("="*80)

print(f"\\n[1] 원본 vs 편집/복원")
print(f"  Original vs Edited:                           {clip_sim_original_vs_edited:.4f}")
print(f"  Original vs Reconstructed (원본 워터마크):      {clip_sim_original_vs_reconstructed:.4f}")
print(f"  Original vs Decoded Reconstructed (편집 워터마크): {clip_sim_original_vs_decoded_reconstructed:.4f}")

print(f"\\n[2] 편집된 이미지 vs 복원")
print(f"  Edited vs Reconstructed (원본 워터마크):        {clip_sim_edited_vs_reconstructed:.4f}")
print(f"  Edited vs Decoded Reconstructed (편집 워터마크):  {clip_sim_edited_vs_decoded_reconstructed:.4f}")

print(f"\\n[3] 두 복원 CLIP 비교")
print(f"  원본 워터마크 복원 vs 편집 워터마크 복원:         {clip_sim_original_reconstructed_vs_decoded_reconstructed:.4f}")

print(f"\\n[Interpretation]")
print(f"  - InstructPix2Pix 변형 정도: {1 - clip_sim_original_vs_edited:.4f}")
print(f"  - 워터마크 bit accuracy: {bit_accuracy:.1%} ({same_bits}/100 bits)")
print(f"  - 원본 워터마크 → CLIP 복원 정확도: {clip_sim_original_vs_reconstructed:.4f}")
print(f"  - 편집 워터마크 → CLIP 복원 정확도: {clip_sim_edited_vs_decoded_reconstructed:.4f}")
print(f"  - 두 복원 CLIP 간 유사도: {clip_sim_original_reconstructed_vs_decoded_reconstructed:.4f}")
print(f"  - Watermark를 통해 원본 의미 보존 가능!")
print("="*80)

## 22. 최종 t-SNE 시각화: 모든 단계의 CLIP 비교

In [ ]:
# t-SNE 시각화: Training data + Original + Reconstructed + Edited + Decoded Reconstructed
print("\\n최종 t-SNE 계산 중...")

all_for_final_tsne = np.vstack([
    all_embeddings,
    test_clip_embedding.reshape(1, -1),
    test_clip_reconstructed.reshape(1, -1),
    edited_clip_embedding.reshape(1, -1),
    decoded_clip_reconstructed.reshape(1, -1)
])

labels_final_tsne = list(all_labels) + ['test_original', 'test_reconstructed', 'test_edited', 'test_decoded_reconstructed']

print(f"t-SNE 계산 중... (총 {len(all_for_final_tsne)} 샘플)")
final_tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
final_embeddings_2d = final_tsne.fit_transform(all_for_final_tsne)

print(f"✅ t-SNE 완료: {final_embeddings_2d.shape}")

In [ ]:
# 최종 t-SNE 시각화
plt.figure(figsize=(16, 12))

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

# Training data
for category in categories:
    mask = all_labels == category
    plt.scatter(
        final_embeddings_2d[:len(all_labels)][mask, 0],
        final_embeddings_2d[:len(all_labels)][mask, 1],
        c=colors[category],
        marker=markers[category],
        label=f'{category.capitalize()} (training)',
        alpha=0.4,
        s=30,
        edgecolors='white',
        linewidth=0.3
    )

# Test image (original)
plt.scatter(
    final_embeddings_2d[-4, 0],
    final_embeddings_2d[-4, 1],
    c='blue',
    marker='*',
    s=600,
    label='1. Original Image CLIP',
    edgecolors='black',
    linewidth=2,
    zorder=5
)

# Test image (reconstructed from original watermark)
plt.scatter(
    final_embeddings_2d[-3, 0],
    final_embeddings_2d[-3, 1],
    c='cyan',
    marker='*',
    s=600,
    label='2. Reconstructed (original watermark)',
    edgecolors='black',
    linewidth=2,
    zorder=5
)

# Test image (edited with InstructPix2Pix)
plt.scatter(
    final_embeddings_2d[-2, 0],
    final_embeddings_2d[-2, 1],
    c='red',
    marker='*',
    s=600,
    label='3. Edited Image CLIP (Violence)',
    edgecolors='black',
    linewidth=2,
    zorder=5
)

# Test image (reconstructed from edited watermark)
plt.scatter(
    final_embeddings_2d[-1, 0],
    final_embeddings_2d[-1, 1],
    c='orange',
    marker='*',
    s=600,
    label='4. Decoded Reconstructed (edited watermark)',
    edgecolors='black',
    linewidth=2,
    zorder=5
)

# 연결선 (주요 경로)
# 1. Original → Reconstructed (원본 워터마크)
plt.plot(
    [final_embeddings_2d[-4, 0], final_embeddings_2d[-3, 0]],
    [final_embeddings_2d[-4, 1], final_embeddings_2d[-3, 1]],
    'cyan',
    linestyle='--',
    linewidth=2,
    alpha=0.6,
    label=f'1→2 (Sim: {clip_sim_original_vs_reconstructed:.3f})'
)

# 2. Original → Edited
plt.plot(
    [final_embeddings_2d[-4, 0], final_embeddings_2d[-2, 0]],
    [final_embeddings_2d[-4, 1], final_embeddings_2d[-2, 1]],
    'red',
    linestyle='--',
    linewidth=2,
    alpha=0.6,
    label=f'1→3 (Sim: {clip_sim_original_vs_edited:.3f})'
)

# 3. Edited → Decoded Reconstructed (편집 워터마크)
plt.plot(
    [final_embeddings_2d[-2, 0], final_embeddings_2d[-1, 0]],
    [final_embeddings_2d[-2, 1], final_embeddings_2d[-1, 1]],
    'orange',
    linestyle='--',
    linewidth=2,
    alpha=0.6,
    label=f'3→4 (Sim: {clip_sim_edited_vs_decoded_reconstructed:.3f})'
)

# 4. 두 복원 CLIP 비교
plt.plot(
    [final_embeddings_2d[-3, 0], final_embeddings_2d[-1, 0]],
    [final_embeddings_2d[-3, 1], final_embeddings_2d[-1, 1]],
    'purple',
    linestyle=':',
    linewidth=1.5,
    alpha=0.5,
    label=f'2↔4 (Sim: {clip_sim_original_reconstructed_vs_decoded_reconstructed:.3f})'
)

plt.title('Final t-SNE: Training Data + Original + VAE Reconstructed + Edited',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=10, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('/content/final_tsne_all_stages.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 최종 t-SNE 시각화 저장: /content/final_tsne_all_stages.png")

## 23. 이미지 품질 메트릭 (PSNR, SSIM, LPIPS)

In [ ]:
# 이미지 품질 메트릭 함수 정의 (demo_war와 동일)
import cv2
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def compute_psnr_ssim(img1, img2):
    """PSNR과 SSIM 계산"""
    img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
    img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
    psnr_value = psnr(img1, img2)
    ssim_value, _ = ssim(img1, img2, full=True, channel_axis=2)
    return psnr_value, ssim_value

def compute_lpips(img1, img2, loss_fn, device):
    """LPIPS 계산"""
    img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
    img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
    img1_tensor = transforms.ToTensor()(img1) * 2 - 1
    img2_tensor = transforms.ToTensor()(img2) * 2 - 1
    img1_tensor = img1_tensor.to(device)
    img2_tensor = img2_tensor.to(device)
    lpips_value = loss_fn(img1_tensor, img2_tensor)
    return lpips_value.item()

print("✅ 이미지 품질 메트릭 함수 정의 완료")

In [ ]:
# LPIPS 모델 로드 및 이미지 메트릭 계산
loss_fn_alex = lpips.LPIPS(net='alex').to(device)

# 이미지를 임시 파일로 저장 (원본과 동일한 크기로)
input_image_pil.save('/content/temp_original.png')
watermarked_pil.save('/content/temp_watermarked.png')
edited_image.save('/content/temp_edited.png')

# 이미지 로드
original_img_cv = cv2.imread('/content/temp_original.png', cv2.IMREAD_COLOR)
watermarked_img_cv = cv2.imread('/content/temp_watermarked.png', cv2.IMREAD_COLOR)
edited_img_cv = cv2.imread('/content/temp_edited.png', cv2.IMREAD_COLOR)

# 메트릭 계산
print("\\n" + "="*80)
print("IMAGE QUALITY METRICS")
print("="*80)

print("\\n[1] Original vs Watermarked (VINE)")
psnr_ow, ssim_ow = compute_psnr_ssim(original_img_cv, watermarked_img_cv)
lpips_ow = compute_lpips(original_img_cv, watermarked_img_cv, loss_fn_alex, device)
print(f"  PSNR:  {psnr_ow:.2f} dB")
print(f"  SSIM:  {ssim_ow:.4f}")
print(f"  LPIPS: {lpips_ow:.4f}")

print("\\n[2] Original vs Edited (InstructPix2Pix)")
psnr_oe, ssim_oe = compute_psnr_ssim(original_img_cv, edited_img_cv)
lpips_oe = compute_lpips(original_img_cv, edited_img_cv, loss_fn_alex, device)
print(f"  PSNR:  {psnr_oe:.2f} dB")
print(f"  SSIM:  {ssim_oe:.4f}")
print(f"  LPIPS: {lpips_oe:.4f}")

print("\\n[3] Watermarked vs Edited")
psnr_we, ssim_we = compute_psnr_ssim(watermarked_img_cv, edited_img_cv)
lpips_we = compute_lpips(watermarked_img_cv, edited_img_cv, loss_fn_alex, device)
print(f"  PSNR:  {psnr_we:.2f} dB")
print(f"  SSIM:  {ssim_we:.4f}")
print(f"  LPIPS: {lpips_we:.4f}")

print("\\n[Interpretation]")
print(f"  - VINE 워터마킹은 시각적으로 거의 무손실 (PSNR={psnr_ow:.1f}dB, SSIM={ssim_ow:.3f})")
print(f"  - InstructPix2Pix 편집으로 이미지가 크게 변형 (SSIM={ssim_oe:.3f})")
print(f"  - 하지만 워터마크는 {bit_accuracy:.1%} 복원됨!")
print(f"  - VAE 기반 워터마크는 강력한 robustness를 보여줌")
print("="*80)

## 24. 최종 요약 및 결론

In [ ]:
print("\\n" + "="*80)
print(" "*25 + "실험 결과 최종 요약")
print("="*80)

print("\\n[1] Training Data")
print(f"  - 데이터셋: semantic_wm")
print(f"  - 전체 샘플: {len(all_embeddings)}")
for cat, count in category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(all_embeddings)*100:.1f}%)")

print("\\n[2] VAE 모델")
print(f"  - Architecture: CLIPCompressionVAE")
print(f"  - Input: CLIP 512D")
print(f"  - Latent: 100D")
print(f"  - Output: Reconstructed CLIP 512D")
print(f"  - Best Val Loss: {min(training_history['val_losses']):.4f}")
print(f"  - Final Cosine Sim: {training_history['val_cosine_similarities'][-1]:.4f}")

print("\\n[3] Test Image (0.png)")
print(f"  - Path: {input_path}")
print(f"  - Size: {input_image_pil.size}")
print(f"  - CLIP Embedding: {test_clip_embedding.shape}")
print(f"  - Latent: {test_latent.shape}")
print(f"  - Watermark: 100 bits ({np.sum(test_watermark)} ones, {100-np.sum(test_watermark)} zeros)")

print("\\n[4] VINE Watermarking")
print(f"  - Encoder: VINE-B-Enc")
print(f"  - Decoder: VINE-B-Dec")
print(f"  - Watermark successfully embedded into image")
print(f"  - Image quality preserved (PSNR={psnr_ow:.1f}dB, SSIM={ssim_ow:.3f})")

print("\\n[5] InstructPix2Pix Transformation")
print(f"  - Prompt: '{edit_prompt}'")
print(f"  - Image semantics changed significantly (SSIM={ssim_oe:.3f})")
print(f"  - CLIP similarity: Original vs Edited = {clip_sim_original_vs_edited:.4f}")

print("\\n[6] Watermark Recovery")
print(f"  - Bit Accuracy: {bit_accuracy:.2%} ({same_bits}/100 bits)")
if bit_accuracy >= 0.95:
    print(f"  - ✓ Excellent recovery rate!")
elif bit_accuracy >= 0.80:
    print(f"  - ✓ Good recovery rate")
else:
    print(f"  - △ Moderate recovery rate")

print("\\n[7] CLIP Similarity Analysis")
print(f"  - Original vs Edited:               {clip_sim_original_vs_edited:.4f}")
print(f"  - Original vs Reconstructed (VAE):  {clip_sim_original_vs_reconstructed:.4f}")
print(f"  - Edited vs Reconstructed (VAE):    {clip_sim_edited_vs_reconstructed:.4f}")

print("\\n[8] Key Findings")
print(f"  ✓ VAE successfully compresses CLIP (512D → 100D) with high fidelity")
print(f"  ✓ Latent space shows clear categorical clustering")
print(f"  ✓ 100-bit watermark can be reliably extracted from latent")
print(f"  ✓ VINE embeds watermark invisibly (PSNR={psnr_ow:.1f}dB)")
print(f"  ✓ Watermark survives InstructPix2Pix transformation ({bit_accuracy:.1%} accuracy)")
print(f"  ✓ VAE-based watermark demonstrates strong robustness")

print("\\n[9] Visualizations Generated")
print(f"  - /content/vae_training_loss_visualization.png")
print(f"  - /content/latent_space_tsne.png")
print(f"  - /content/original_vs_watermarked.png")
print(f"  - /content/image_transformation_pipeline.png")
print(f"  - /content/final_tsne_all_stages.png")

print("\\n" + "="*80)
print("✅ 실험 완료! 🎉")
print("="*80)
print("\\nConclusion:")
print("This experiment demonstrates that VAE-based latent watermarking")
print("combined with VINE can achieve robust watermark embedding that")
print("survives semantic image transformations while maintaining high")
print("image quality and semantic preservation through CLIP embeddings.")
print("="*80)

## 25. 카테고리별 CLIP Similarity 분석 (Normal / Violence / Sexual)

In [ ]:
# 카테고리별로 Training data와 Test image 유사도 계산
print("\\n" + "="*80)
print("카테고리별 CLIP Similarity 분석")
print("="*80)

# 각 카테고리별 평균 CLIP embedding
category_mean_embeddings = {}
for category in categories:
    mask = all_labels == category
    category_embeddings = all_embeddings[mask]
    category_mean = category_embeddings.mean(axis=0)
    category_mean_embeddings[category] = category_mean / np.linalg.norm(category_mean)

print("\\n[1] Original Image와 각 카테고리 평균 유사도")
print("-" * 80)
for category in categories:
    sim = np.dot(test_clip_embedding, category_mean_embeddings[category])
    print(f"  {category.capitalize():10s} : {sim:.4f}")

print("\\n[2] Reconstructed Image (VAE)와 각 카테고리 평균 유사도")
print("-" * 80)
for category in categories:
    sim = np.dot(test_clip_reconstructed, category_mean_embeddings[category])
    print(f"  {category.capitalize():10s} : {sim:.4f}")

print("\\n[3] Edited Image (Violence)와 각 카테고리 평균 유사도")
print("-" * 80)
for category in categories:
    sim = np.dot(edited_clip_embedding, category_mean_embeddings[category])
    print(f"  {category.capitalize():10s} : {sim:.4f}")

print("\\n[4] 카테고리별 유사도 변화 (Original → Edited)")
print("-" * 80)
for category in categories:
    sim_original = np.dot(test_clip_embedding, category_mean_embeddings[category])
    sim_edited = np.dot(edited_clip_embedding, category_mean_embeddings[category])
    change = sim_edited - sim_original
    change_percent = (change / sim_original) * 100
    print(f"  {category.capitalize():10s} : {sim_original:.4f} → {sim_edited:.4f} (Δ{change:+.4f}, {change_percent:+.1f}%)")

print("="*80)

In [ ]:
# 카테고리별 변화량을 히트맵으로 시각화
fig, ax = plt.subplots(figsize=(12, 6))

# 변화량 계산
stages = ['1. Original', '2. Reconstructed\n(Original WM)', '3. Edited', '4. Decoded Recon\n(Edited WM)']
similarity_matrix = []

for category in categories:
    row = [
        np.dot(test_clip_embedding, category_mean_embeddings[category]),
        np.dot(test_clip_reconstructed, category_mean_embeddings[category]),
        np.dot(edited_clip_embedding, category_mean_embeddings[category]),
        np.dot(decoded_clip_reconstructed, category_mean_embeddings[category])
    ]
    similarity_matrix.append(row)

similarity_matrix = np.array(similarity_matrix)

# 히트맵 그리기
im = ax.imshow(similarity_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

# 축 레이블
ax.set_xticks(np.arange(len(stages)))
ax.set_yticks(np.arange(len(categories)))
ax.set_xticklabels(stages, fontsize=12)
ax.set_yticklabels([cat.capitalize() for cat in categories], fontsize=12)

# 각 셀에 값 표시
for i in range(len(categories)):
    for j in range(len(stages)):
        text = ax.text(j, i, f'{similarity_matrix[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=11, fontweight='bold')

# 컬러바
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Cosine Similarity', fontsize=12, fontweight='bold')

ax.set_title('Category Similarity Heatmap Across Pipeline', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/category_similarity_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 카테고리별 유사도 히트맵 저장: /content/category_similarity_heatmap.png")

In [ ]:
# 카테고리 예측 (가장 유사한 카테고리)
print("\\n" + "="*80)
print("카테고리 예측 (Highest Similarity)")
print("="*80)

def predict_category(clip_embedding, category_means):
    """가장 유사도가 높은 카테고리 예측"""
    similarities = {}
    for cat in categories:
        sim = np.dot(clip_embedding, category_means[cat])
        similarities[cat] = sim
    predicted = max(similarities, key=similarities.get)
    return predicted, similarities

# Original Image 예측
pred_orig, sims_orig = predict_category(test_clip_embedding, category_mean_embeddings)
print(f"\\n[Original Image]")
print(f"  Predicted Category: {pred_orig.upper()}")
for cat in categories:
    marker = "★" if cat == pred_orig else " "
    print(f"  {marker} {cat.capitalize():10s}: {sims_orig[cat]:.4f}")

# Reconstructed Image 예측
pred_recon, sims_recon = predict_category(test_clip_reconstructed, category_mean_embeddings)
print(f"\\n[Reconstructed Image (VAE)]")
print(f"  Predicted Category: {pred_recon.upper()}")
for cat in categories:
    marker = "★" if cat == pred_recon else " "
    print(f"  {marker} {cat.capitalize():10s}: {sims_recon[cat]:.4f}")

# Edited Image 예측
pred_edited, sims_edited = predict_category(edited_clip_embedding, category_mean_embeddings)
print(f"\\n[Edited Image (Violence)]")
print(f"  Predicted Category: {pred_edited.upper()}")
for cat in categories:
    marker = "★" if cat == pred_edited else " "
    print(f"  {marker} {cat.capitalize():10s}: {sims_edited[cat]:.4f}")

# 예측 변화 요약
print(f"\\n[Category Prediction Summary]")
print(f"  Original:      {pred_orig.upper()}")
print(f"  Reconstructed: {pred_recon.upper()}")
print(f"  Edited:        {pred_edited.upper()}")

if pred_orig == pred_recon:
    print(f"  ✓ VAE reconstruction preserves category ({pred_orig.upper()})")
else:
    print(f"  △ VAE reconstruction changed category: {pred_orig.upper()} → {pred_recon.upper()}")

if pred_orig == pred_edited:
    print(f"  ! Edited image surprisingly remained in same category ({pred_orig.upper()})")
else:
    print(f"  ✓ Edited image changed category as expected: {pred_orig.upper()} → {pred_edited.upper()}")

print("="*80)

In [ ]:
# Test 데이터셋 로드
print("\n" + "="*80)
print("Test 데이터셋 로드 중...")
print("="*80)

test_data_path = '/content/semantic_wm/dataset/test'
test_embeddings_list = []
test_labels_list = []
test_category_stats = {}

categories = ['normal', 'violence', 'sexual']

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")
    category_path = os.path.join(test_data_path, category)

    if not os.path.exists(category_path):
        print(f"  ⚠️ 경로가 존재하지 않습니다: {category_path}")
        continue

    image_files = [f for f in os.listdir(category_path)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"  - 발견된 이미지 수: {len(image_files)}")

    # CLIP embedding 추출 (전체 이미지)
    category_embeddings = []
    for img_file in tqdm(image_files, desc=f"{category} CLIP 추출"):
        img_path = os.path.join(category_path, img_file)
        try:
            image = Image.open(img_path).convert('RGB')
            inputs = clip_processor(images=image, return_tensors="pt").to(device)

            with torch.no_grad():
                embedding = clip_model.get_image_features(**inputs)
                embedding = embedding.cpu().numpy().flatten()

            category_embeddings.append(embedding)
        except Exception as e:
            print(f"  ⚠️ 이미지 로드 실패: {img_file} - {e}")
            continue
    test_embeddings_list.append(np.array(category_embeddings))
    test_labels_list.extend([category] * len(category_embeddings))
    test_category_stats[category] = len(category_embeddings)

    print(f"  ✓ 추출 완료: {len(category_embeddings)} images")

# 데이터 결합
test_embeddings = np.vstack(test_embeddings_list)
test_labels = np.array(test_labels_list)

print("\n" + "="*80)
print("✅ Test 데이터 로드 완료")
print("="*80)
print(f"전체 test embedding shape: {test_embeddings.shape}")
print(f"전체 test label 수: {len(test_labels)}")
print("\n카테고리별 통계:")
for cat, count in test_category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(test_labels)*100:.1f}%)")
print("="*80)

In [ ]:
# Test 데이터셋에 대한 워터마크 추출 및 복원
print("\n" + "="*80)
print("Test 데이터셋 워터마크 추출 및 복원 중...")
print("="*80)

test_embeddings_tensor = torch.FloatTensor(test_embeddings).to(device)

# Step 1: 원본 CLIP → Latent 추출
test_latent_vectors = []
vae_model.eval()
with torch.no_grad():
    batch_size = 64
    for i in range(0, len(test_embeddings_tensor), batch_size):
        batch = test_embeddings_tensor[i:i+batch_size]
        mu, _ = vae_model.encode(batch)
        test_latent_vectors.append(mu.cpu())

test_latent_vectors = torch.cat(test_latent_vectors, dim=0).numpy()
print(f"✅ Step 1: Latent 추출 완료 - shape: {test_latent_vectors.shape}")

# Step 2: Latent → Watermark 추출 (100-bit)
test_watermarks = []
for latent in test_latent_vectors:
    watermark = watermarker.latent_to_watermark(latent)
    test_watermarks.append(watermark)
test_watermarks = np.array(test_watermarks)
print(f"✅ Step 2: Watermark 추출 완료 - shape: {test_watermarks.shape}")

# Step 3: Watermark → Latent 복원
test_latent_restored = []
for watermark in test_watermarks:
    latent_restored = watermarker.watermark_to_latent(watermark, latent_stats)
    test_latent_restored.append(latent_restored)
test_latent_restored = np.array(test_latent_restored)
print(f"✅ Step 3: Latent 복원 완료 - shape: {test_latent_restored.shape}")

# Step 4: 복원된 Latent → CLIP 복원
test_latent_restored_tensor = torch.FloatTensor(test_latent_restored).to(device)
reconstructed_embeddings = []

with torch.no_grad():
    batch_size = 64
    for i in range(0, len(test_latent_restored_tensor), batch_size):
        batch = test_latent_restored_tensor[i:i+batch_size]
        recon = vae_model.decode(batch)
        reconstructed_embeddings.append(recon.cpu())

reconstructed_embeddings = torch.cat(reconstructed_embeddings, dim=0).numpy()
print(f"✅ Step 4: CLIP 복원 완료 - shape: {reconstructed_embeddings.shape}")

print("\n" + "="*80)
print("✅ 전체 워터마크 파이프라인 완료")
print("="*80)
print(f"   원본 CLIP → Latent → Watermark (100-bit) → Latent → 복원 CLIP")
print(f"   - 원본 CLIP shape: {test_embeddings.shape}")
print(f"   - 복원 CLIP shape: {reconstructed_embeddings.shape}")
print("="*80)

In [ ]:
# Cosine Similarity 계산
print("\n" + "="*80)
print("Cosine Similarity 분석")
print("="*80)

cosine_sims = []
for orig, recon in zip(test_embeddings, reconstructed_embeddings):
    # Normalize
    orig_norm = orig / (np.linalg.norm(orig) + 1e-8)
    recon_norm = recon / (np.linalg.norm(recon) + 1e-8)
    # Cosine similarity
    cos_sim = np.dot(orig_norm, recon_norm)
    cosine_sims.append(cos_sim)

avg_cosine_sim = np.mean(cosine_sims)
std_cosine_sim = np.std(cosine_sims)
min_cosine_sim = np.min(cosine_sims)
max_cosine_sim = np.max(cosine_sims)

print(f"\n전체 통계:")
print(f"  - 평균 Cosine Similarity: {avg_cosine_sim:.6f}")
print(f"  - 표준편차: {std_cosine_sim:.6f}")
print(f"  - 최소값: {min_cosine_sim:.6f}")
print(f"  - 최대값: {max_cosine_sim:.6f}")

# 카테고리별 Cosine Similarity
print(f"\n카테고리별 통계:")
for category in categories:
    mask = test_labels == category
    category_sims = [cosine_sims[i] for i in range(len(cosine_sims)) if mask[i]]
    if len(category_sims) > 0:
        cat_mean = np.mean(category_sims)
        cat_std = np.std(category_sims)
        print(f"  - {category.capitalize():10s}: {cat_mean:.6f} ± {cat_std:.6f}")

print("="*80)

### 13.6. Test 데이터셋 t-SNE 시각화 (원본 vs 복원)

In [ ]:
# t-SNE 시각화: 원본 CLIP vs 복원 CLIP
print("\n" + "="*80)
print("t-SNE 시각화 준비 중...")
print("="*80)

from sklearn.manifold import TSNE

# 색상 맵핑
category_colors = {
    'normal': 'blue',
    'violence': 'red',
    'sexual': 'green'
}
label_colors = [category_colors[label] for label in test_labels]

# t-SNE 계산
print("\n원본 CLIP embedding t-SNE 계산 중...")
tsne_original = TSNE(n_components=2, random_state=42, perplexity=30)
original_tsne = tsne_original.fit_transform(test_embeddings)

print("복원 CLIP embedding t-SNE 계산 중...")
tsne_recon = TSNE(n_components=2, random_state=42, perplexity=30)
recon_tsne = tsne_recon.fit_transform(reconstructed_embeddings)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 원본 CLIP
ax = axes[0]
for category in categories:
    mask = test_labels == category
    ax.scatter(original_tsne[mask, 0], original_tsne[mask, 1],
              c=category_colors[category], label=category.capitalize(),
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.set_title('Original CLIP Embeddings (Test Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)

# 복원 CLIP
ax = axes[1]
for category in categories:
    mask = test_labels == category
    ax.scatter(recon_tsne[mask, 0], recon_tsne[mask, 1],
              c=category_colors[category], label=category.capitalize(),
              alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.set_title(f'Reconstructed CLIP from Watermark (Test Set)',
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
ax.set_ylabel('t-SNE Dimension 2', fontsize=11)

plt.suptitle('Test Dataset: Original vs Watermark-Reconstructed CLIP Embeddings',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/test_clip_comparison_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ t-SNE 시각화 완료")
print(f"   저장 위치: /content/test_clip_comparison_tsne.png")
print("="*80)

In [ ]:
# Cosine Similarity 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 전체 분포
ax = axes[0]
ax.hist(cosine_sims, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(avg_cosine_sim, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_cosine_sim:.4f}')
ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Overall Cosine Similarity Distribution (Test Set)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 카테고리별 분포
ax = axes[1]
for category in categories:
    mask = test_labels == category
    category_sims = [cosine_sims[i] for i in range(len(cosine_sims)) if mask[i]]
    ax.hist(category_sims, bins=30, alpha=0.5, label=category.capitalize(),
           color=category_colors[category], edgecolor='black')

ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Category-wise Cosine Similarity Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/test_cosine_similarity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Cosine Similarity 분포 시각화 완료")
print(f"   저장 위치: /content/test_cosine_similarity_distribution.png")

In [ ]:
# 종합 분석 및 결론
print("\n" + "="*80)
print("TEST 데이터셋 종합 분석 결과")
print("="*80)

print("\n📊 성능 요약:")
print(f"   • 평균 Cosine Similarity: {avg_cosine_sim:.6f}")
print(f"   • 표준편차: {std_cosine_sim:.6f}")
print(f"   • 95% 신뢰구간: [{avg_cosine_sim - 1.96*std_cosine_sim:.6f}, {avg_cosine_sim + 1.96*std_cosine_sim:.6f}]")

print("\n📈 카테고리별 성능:")
for category in categories:
    mask = test_labels == category
    category_sims = [cosine_sims[i] for i in range(len(cosine_sims)) if mask[i]]
    if len(category_sims) > 0:
        cat_mean = np.mean(category_sims)
        cat_std = np.std(category_sims)
        cat_min = np.min(category_sims)
        cat_max = np.max(category_sims)
        print(f"\n   [{category.capitalize()}]")
        print(f"      - 평균: {cat_mean:.6f} ± {cat_std:.6f}")
        print(f"      - 범위: [{cat_min:.6f}, {cat_max:.6f}]")
        print(f"      - 샘플 수: {len(category_sims)}")

print("\n💡 해석:")
if avg_cosine_sim > 0.95:
    print("   ✅ 매우 우수한 복원 품질 (Cosine Sim > 0.95)")
    print("      → 워터마크에서 원본 CLIP의 semantic 정보를 거의 완벽하게 보존")
elif avg_cosine_sim > 0.90:
    print("   ✅ 우수한 복원 품질 (Cosine Sim > 0.90)")
    print("      → 워터마크가 원본 CLIP의 semantic 정보를 잘 보존")
elif avg_cosine_sim > 0.80:
    print("   ⚠️ 보통 복원 품질 (Cosine Sim > 0.80)")
    print("      → 일부 semantic 정보 손실이 있으나 전반적으로 유지됨")
else:
    print("   ❌ 낮은 복원 품질 (Cosine Sim < 0.80)")
    print("      → 상당한 semantic 정보 손실")

print("\n✅ Test 데이터셋 분석 완료!")
print("="*80)